# Stage 03: Python Fundamentals

Stage 03. **I based it on the lecture notebook.**

In [ ]:
# Install missing packages (uncomment and run to install).
# !pip install numpy pandas matplotlib

In [ ]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "prismatic_evoluations_prices.csv").exists() and (
    ROOT.parent / "data" / "raw" / "prismatic_evoluations_prices.csv"
).exists():
    ROOT = ROOT.parent

CHECKS = [
    ("data/raw/prismatic_evoluations_prices.csv", "NEEDED", "sample Prismatic Evolutions prices"),
    ("src/utils.py", "NEEDED", "helper module the notebook imports"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Raw CSV.
RAW_CSV = ROOT / "data" / "raw" / "prismatic_evoluations_prices.csv"
# Processed outputs.
PROCESSED_DIR = ROOT / "data" / "processed"

## NumPy operations

In [ ]:
# Sample card prices.
sample_prices = np.array([145.0, 98.0, 72.0, 18.5, 0.25])
pct_of_top = sample_prices / sample_prices.max() * 100

print("Sample card prices:", sample_prices)
print("As % of top card:", pct_of_top.round(1))

## Loop vs. vectorization

In [ ]:
big_array = np.arange(1_000_000)

# Loop vs vectorized * 2.
%timeit [x * 2 for x in big_array]
%timeit big_array * 2

## Load and inspect dataset

In [ ]:
# Load sample prices.
df = pd.read_csv(RAW_CSV)
df["date"] = pd.to_datetime(df["date"])

display(df.head())
df.info()
df.select_dtypes(include="number").describe()

## Summary statistics and groupby

In [ ]:
# Mean price by rarity.
summary = df.groupby("rarity", as_index=False)["market_price"].mean().round(2)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
summary_path = PROCESSED_DIR / "summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Saved: {summary_path}")
summary

## Rolling returns

In [ ]:
from src.utils import add_rolling_return, get_summary_stats

# 3-day rolling % change.
df_features = add_rolling_return(df, window=3)
df_features[["card_name", "date", "market_price", "rolling_return"]].tail(8)

## Utility function

In [ ]:
# Numeric summary stats.
get_summary_stats(df)

## Bonus plot

In [ ]:
# Latest price by card.
latest = df.sort_values("date").groupby("card_name", as_index=False).tail(1)
latest = latest.sort_values("market_price", ascending=False)

plt.barh(latest["card_name"], latest["market_price"])
plt.title("Latest Prismatic Evolutions prices (sample)")
plt.xlabel("market_price")
plt.gca().invert_yaxis()

plot_path = PROCESSED_DIR / "price_barplot.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")